# AIMarx TRAIN-03 — Qwen3-0.6B Colab smoke

Free-tier only. Select a GPU runtime, then run cells in order. The notebook never purchases compute, uploads to Hugging Face, or exports the 20 smoke-test cases.

In [ ]:
import os, subprocess, sys
assert os.path.exists('/content'), 'Run this notebook in Google Colab'
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
REPO = 'https://github.com/hongkhang21998-creator/AIMarx.git'
BRANCH = 'codex/train03-colab-qwen06'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, '/content/AIMarx'], check=True)
os.chdir('/content/AIMarx')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/colab_qwen06/requirements-colab.txt'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.prepare', '/content/aimarx-colab-data'], check=True)

## Phase 1: one optimizer step and a complete checkpoint
This is a separate Python process. It fails closed if Colab did not allocate a CUDA GPU with at least 12 GiB.

In [ ]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-colab-data', '--output', '/content/aimarx-colab-output', '--stop-after', '1'], check=True)

## Phase 2: new process resumes checkpoint 1 and reaches five total steps

In [ ]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-colab-data', '--output', '/content/aimarx-colab-output', '--resume-from', '/content/aimarx-colab-output/checkpoint-1'], check=True)

In [ ]:
import json, pathlib, shutil
final_manifest = json.loads(pathlib.Path('/content/aimarx-colab-output/checkpoint-5/aimarx-manifest.json').read_text())
assert final_manifest['global_step'] == 5
archive = shutil.make_archive('/content/AIMarx-Qwen3-0.6B-LoRA-smoke-step5', 'zip', '/content/aimarx-colab-output')
print(archive, final_manifest)
from google.colab import files
files.download(archive)